# Capstone: Quantum-kernel classifier within hardware limits

**Quantum Computing Advanced · Module 6 · about 5 hours** · notebook version 2026-10-10

**The question of this experiment.** Does the quantum-kernel classifier of Module 4 still work when every kernel entry comes from a noisy quantum computer with the limits of a real run (8 training points, 4 test points, 1,000 shots per entry), and what can 4 test points tell you about it?

You run the experiment on a **noisy simulator**, a simulator that adds the errors of a real quantum computer described by a **noise model**. You compare what you measure with what the noise model predicts, say how certain every result is, and say what the experiment cannot tell you. The capstone brings together the code, the mitigation and the application of Modules 2, 3 and 4, and you write more of the code yourself than in a lab.

**The plan.**

1. write the statistics functions every capstone uses (Step 1) and meet the capstone noise model (Step 2);
2. build the **overlap circuit** and estimate one **kernel entry** from 1,000 shots (Step 3);
3. build the **training and test kernels** with one circuit per entry (Step 4);
4. train the support vector classifier on exact, noisy and shot-based kernels, and compare with a classical kernel (Step 5);
5. find out **why the noise does so little harm**, and what 4 test points can and cannot show (Step 6);
6. get your verification value (Step 7).

**How it is graded.** Your capstone quiz in Canvas (the one for this experiment, 20 points, pass at 16) asks for your verification value from the check cell (6 points), three results from this notebook and four questions about uncertainty, interpretation and limits (2 points each). Question 1 gives you three personal numbers: PAIR, P2 and READOUT. At the end you write a short **summary** of your experiment (required, not graded), which you submit in the Course Completion module.

**Saving your work.** JupyterLite keeps your changes in this browser, so you can stop and come back. To keep a copy elsewhere, use **File → Download**. Run each code cell with **Shift + Enter**, in order; if you come back later, run the cells from the top again.

## Step 0: start Python

Run the cell below. The first time, Python can take up to a minute to start in your browser, and Matplotlib takes a few more seconds to load.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import qsim
from qsim import QuantumCircuit, NoiseModel, ReadoutError, depolarizing_error, simulate_density_matrix
from sklearn.svm import SVC

print("Ready. qsim", qsim.__version__, "| NumPy", np.__version__)

## Step 1: how certain is a result?

In this capstone two kinds of numbers have an uncertainty, and they get it from different places. A **kernel entry** is a fraction of shots, so its uncertainty comes from the shots: the formulas below are for these. A classifier's **accuracy** is a fraction of **test points**: its uncertainty comes from how many points you test, not from the shots, and two classifiers tested on the same points are compared point by point (Step 6), not with the 3σ rule below.

**A fraction.** If a result has probability p and you take S shots, the fraction f of shots with that result has standard deviation

**σ = √( p (1 − p) / S )**.

**An expectation value of ±1 results**, such as ⟨Z⟩ or ⟨X X X⟩: each shot gives +1 or −1, and the average E has standard deviation

**σ = √( (1 − E²) / S )**.

This is twice the σ of the fraction of +1 results, because E = 2 f − 1. For E = 0.8 and S = 4,000 it is √(0.36 / 4000) = 0.0095.

**A combination of independent results.** Mitigation combines several measured values: an estimate Ê = c₁ E₁ + c₂ E₂ + ... from independent runs with uncertainties σ₁, σ₂, ... has

**σ = √( (c₁ σ₁)² + (c₂ σ₂)² + ... )**.

Uncertainties add **in quadrature**. The difference of two results is the special case c = (1, −1): σ_diff = √(σ₁² + σ₂²). Large coefficients, as in an extrapolation, make the combination much less certain than any single run: this is the price of mitigation you met in Module 3.

**The course rule: 3σ.** A measured value agrees with an expected value when they differ by at most 3σ. A difference larger than 3σ_diff is **significant**: shot noise alone would almost never produce it. A smaller difference may be real, but your data cannot show it. Two cautions: the rule treats the spread as normal (bell-shaped), which is good when many shots give each result; and when you compare many settings, chance alone makes one "significant" difference likely somewhere (with 20 comparisons, about 5%). Treat a single surprising difference among many as a question to test again with new shots.

**Accuracies are reported differently.** Give an accuracy as a count, for example 4 of 4 or 18 of 20. Do not put an observed 4 of 4 into √(p (1 − p) / N): p = 1 gives σ = 0, a certainty that 4 points cannot give. If you need an interval, use one made for small counts, such as the Wilson interval (Step 6).

**Your task:** write the three functions below. Every later step uses them.

- `fraction_sigma(p, shots)`: √(p (1 − p) / shots).
- `expectation_sigma(E, shots)`: √((1 − E²) / shots).
- `combined_sigma(coefficients, sigmas)`: √(Σ (cᵢ σᵢ)²) for two lists of the same length.

In [ ]:
def fraction_sigma(p, shots):
    """The standard deviation of a fraction measured with `shots` shots, when the true probability is p."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete fraction_sigma() first.")


def expectation_sigma(E, shots):
    """The standard deviation of the average of `shots` results of +1 or -1 with expectation value E."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete expectation_sigma() first.")


def combined_sigma(coefficients, sigmas):
    """The standard deviation of sum(c_i * E_i) for independent E_i with standard deviations sigma_i."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete combined_sigma() first.")

In [ ]:
print("fraction_sigma(0.0613, 4000) =", round(fraction_sigma(0.0613, 4000), 5), "  (expected 0.00379)")
print("expectation_sigma(0.8, 4000) =", round(expectation_sigma(0.8, 4000), 5), "  (expected 0.00949)")
print("combined_sigma([1, -1], [0.01, 0.02]) =", round(combined_sigma([1, -1], [0.01, 0.02]), 5), "  (expected 0.02236)")
print("combined_sigma([1.875, -1.25, 0.375], [0.012, 0.015, 0.02]) =",
      round(combined_sigma([1.875, -1.25, 0.375], [0.012, 0.015, 0.02]), 5), "  (expected 0.03023)")

## Step 2: the capstone noise model

The noise model of the capstone is the one you used in Level 2 Module 6, with one addition:

- after every **two-qubit gate**, a **depolarizing error** with parameter P2 (with probability P2 the two qubits are replaced by a completely random state);
- after every **one-qubit gate**, the same on one qubit with P2 / 10;
- an **idle error** on `id` gates, used only by the repetition-code capstone (a qubit that waits while others are measured);
- at **readout**, a 0 is recorded as 1 with probability READOUT and a 1 as 0 with probability 2 · READOUT.

The course values are **P2 = 0.01** and **READOUT = 0.02**, several times worse than IBM's best devices, so that the effects show clearly in a few thousand shots. Your capstone quiz gives you your own P2 and READOUT (0.005 to 0.030 each) for the check cell.

The prepared cell builds the model and three tools:

- `exact_probabilities(qc, p2, readout)`: the noise model's **exact** probability of every recorded result, from the density matrix (Module 1) and the readout confusion; no shots, no randomness. This is the expected value the 3σ rule compares with.
- `sample_counts(probs, shots, seed)`: random shots drawn from those probabilities. A run of the noisy simulator gives counts with exactly these statistics; drawing them from the exact probabilities is much faster in the browser, and the seed makes your run repeatable.
- `apply_readout(probs, readout)`: the readout errors alone, applied to probabilities.

In [ ]:
ONE_QUBIT_GATES = ["h", "x", "y", "z", "rx", "ry", "rz", "p", "s", "sdg", "t", "tdg", "sx"]
TWO_QUBIT_GATES = ["cx", "cz", "cp", "rzz", "swap"]
P2_COURSE, READOUT_COURSE = 0.01, 0.02


def confusion(readout):
    """Prepared. Rows: the true bit (0, 1); columns: the recorded bit."""
    return np.array([[1 - readout, readout], [2 * readout, 1 - 2 * readout]])


def capstone_noise_model(p2=P2_COURSE, readout=READOUT_COURSE, idle=0.0):
    """Prepared: depolarizing error p2 after two-qubit gates, p2/10 after one-qubit gates, idle on id gates, readout."""
    nm = NoiseModel()
    if p2 > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(p2 / 10, 1), ONE_QUBIT_GATES)
        nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), TWO_QUBIT_GATES)
    if idle > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(idle, 1), ["id"])
    if readout > 0:
        nm.add_all_qubit_readout_error(ReadoutError(confusion(readout).tolist()))
    return nm


def apply_readout(probs, readout):
    """Prepared. probs over m recorded bits (index bit j = classical bit j); each bit is flipped by the confusion."""
    probs = np.asarray(probs, dtype=float)
    m = int(round(math.log2(len(probs))))
    c = confusion(readout)
    for j in range(m):
        t = probs.reshape(2 ** (m - 1 - j), 2, 2 ** j)
        probs = np.einsum("aib,ij->ajb", t, c).reshape(-1)
    return probs


def exact_probabilities(qc, p2=P2_COURSE, readout=READOUT_COURSE, idle=0.0):
    """Prepared: the noise model's exact probability of each recorded result, {"bits": p}, Qiskit's key order.
    qc must end with its measurements (each classical bit receives one qubit). No shots are taken."""
    measured = {inst.clbits[0]: inst.qubits[0] for inst in qc.data if inst.name == "measure"}
    body = qc.copy()
    body.remove_final_measurements()
    nm = capstone_noise_model(p2, 0.0, idle)
    rho = simulate_density_matrix(body, nm if not nm.is_ideal() else None)   # the exact mixed state after the noisy gates
    m = len(measured)
    probs = rho.probabilities(qargs=[measured[j] for j in range(m)])
    probs = apply_readout(probs, readout)                                       # then the readout errors
    return {format(i, f"0{m}b"): float(p) for i, p in enumerate(probs) if p > 1e-15}


def sample_counts(probs, shots, seed):
    """Prepared: `shots` random results drawn from exact probabilities, as a quantum computer gives them."""
    keys = sorted(probs)
    p = np.array([probs[k] for k in keys])
    draw = np.random.default_rng(seed).multinomial(shots, p / p.sum())
    return {k: int(c) for k, c in zip(keys, draw) if c > 0}


print(capstone_noise_model())

## Step 3: one kernel entry

In Module 4 Track A, the **ZZ feature map** U(x) put a data point x = (x₁, x₂) into a two-qubit state |φ(x)⟩ = U(x)|00⟩, and the **quantum kernel** was the overlap

K(x, x′) = |⟨φ(x′)|φ(x)⟩|².

A quantum computer measures it with the **overlap circuit**: apply U(x), then U(x′)⁻¹, and measure. The probability of reading **00** is |⟨00|U(x′)⁻¹U(x)|00⟩|² = K(x, x′). So K is the fraction of shots that give 00, a fraction with σ = √(K(1 − K)/shots).

The prepared cell defines `feature_map(x)` (the Module 4 map, two repetitions) and loads the data: 8 training points (4 of each class) and 4 test points (2 of each) from the Module 4 data set.

**Your task:**

- `overlap_circuit(x1, x2)`: a QuantumCircuit with 2 qubits and 2 classical bits: `feature_map(x1)`, then `feature_map(x2).inverse()` (use `qc.compose(..., inplace=True)`), then measure qubit i into classical bit i.
- `kernel_from_counts(counts)`: the fraction of shots (or probability) with every bit 0. Divide by the total; a key that never appeared is missing from the counts.

In [ ]:
import json


def feature_map(x, reps=2):
    """Prepared: the ZZ feature map of Module 4 (H, P(2 x_i), and CX-P-CX with angle 2 (pi - x_1)(pi - x_2))."""
    x = [float(v) for v in x]
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        for i in range(n):
            qc.h(i)
        for i in range(n):
            qc.p(2 * x[i], i)
        for i in range(n):
            for j in range(i + 1, n):
                qc.cx(i, j)
                qc.p(2 * (np.pi - x[i]) * (np.pi - x[j]), j)
                qc.cx(i, j)
    return qc


def pick(X, y, k):
    """Prepared: the first k points of each class."""
    X, y = np.array(X), np.array(y)
    idx = [int(i) for c in sorted(set(y.tolist())) for i in np.where(y == c)[0][:k]]
    return X[idx], y[idx]


DATA = json.load(open("l3_m4a_data.json"))["adhoc"]
X_train, y_train = pick(DATA["X_train"], DATA["y_train"], 4)
X_test, y_test = pick(DATA["X_test"], DATA["y_test"], 2)
print("training points:", len(X_train), " classes", y_train.tolist())
print("test points:    ", len(X_test), " classes", y_test.tolist())

In [ ]:
def overlap_circuit(x1, x2):
    """feature_map(x1), then feature_map(x2).inverse(), then qubit i measured into classical bit i."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete overlap_circuit() first.")


def kernel_from_counts(counts):
    """The fraction of shots (or the probability) with every bit 0."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete kernel_from_counts() first.")

In [ ]:
SHOTS_K = 1000
x0, x1 = X_train[0], X_train[1]
print(overlap_circuit(x0, x1).draw())
k_ideal = kernel_from_counts(exact_probabilities(overlap_circuit(x0, x1), 0, 0))
k_noisy = kernel_from_counts(exact_probabilities(overlap_circuit(x0, x1)))
k_run = kernel_from_counts(sample_counts(exact_probabilities(overlap_circuit(x0, x1)), SHOTS_K, seed=1))
s = fraction_sigma(k_noisy, SHOTS_K)
print(f"K(x0, x1): ideal {k_ideal:.4f}   noisy, exact {k_noisy:.4f}   your run of {SHOTS_K} shots {k_run:.4f} +- {s:.4f}")
print(f"K(x1, x0): ideal {kernel_from_counts(exact_probabilities(overlap_circuit(x1, x0), 0, 0)):.4f}"
      f"   noisy, exact {kernel_from_counts(exact_probabilities(overlap_circuit(x1, x0))):.4f}")
print(f"K(x0, x0): ideal {kernel_from_counts(exact_probabilities(overlap_circuit(x0, x0), 0, 0)):.4f}"
      f"   noisy, exact {kernel_from_counts(exact_probabilities(overlap_circuit(x0, x0))):.4f}")

**What to notice.** The ideal K(x₀, x₁) is 0.6017; the noise lowers it to **0.5583**, a bias of about 0.043, more than twice the shot noise of 1,000 shots. Three effects of the noise show here:

- **The diagonal is not 1.** K(x, x) is 1 in theory, but the noisy circuit gives about **0.897**: the circuit still has all its gates, and they make errors.
- **The kernel is not symmetric.** K(x₁, x₀) = 0.5496 differs from K(x₀, x₁) = 0.5583. The two circuits are inverses of each other and meet the noise at different places; the readout error is not symmetric either.
- **Shot noise.** With 1,000 shots every entry has σ up to √(0.25 / 1000) ≈ 0.016.

## Step 4: the kernel matrices

The classifier needs two matrices:

- the **training kernel**, 8 × 8, K[i][j] = K(xᵢ, xⱼ) for the training points;
- the **test kernel**, 4 × 8, one row per test point and one column per training point.

Every entry is a circuit, and on hardware every circuit costs time. Two savings are standard: the **diagonal is set to 1** (K(x, x) = 1 needs no circuit), and the training kernel is taken as **symmetric**, so only the pairs i < j are run (one circuit each) and each value is copied to [j][i]. That makes 8 · 7 / 2 = 28 training circuits and 4 · 8 = 32 test circuits: **60 circuits** of 1,000 shots.

**Your task:** write both functions for any estimator `estimate(x1, x2)` that returns one entry:

- `training_kernel(X, estimate)`: a len(X) × len(X) NumPy array with 1 on the diagonal and `estimate(X[i], X[j])` for i < j, copied to [j][i]: one call per pair, no more;
- `test_kernel(X_test, X_train, estimate)`: the len(X_test) × len(X_train) array of `estimate(X_test[a], X_train[b])`.

The cell after them builds three versions: **exact** (no noise), **noisy exact** (the noise model's exact probabilities) and **shots** (1,000 shots per circuit, the kernel a real run would give).

In [ ]:
def training_kernel(X, estimate):
    """len(X) x len(X): 1 on the diagonal, estimate(X[i], X[j]) for i < j, copied to [j][i]."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete training_kernel() first.")


def test_kernel(X_test, X_train, estimate):
    """len(X_test) x len(X_train): estimate(X_test[a], X_train[b])."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete test_kernel() first.")

In [ ]:
calls = {"n": 0}


def est_exact(a, b):
    return kernel_from_counts(exact_probabilities(overlap_circuit(a, b), 0, 0))


def est_noisy(a, b):
    return kernel_from_counts(exact_probabilities(overlap_circuit(a, b)))


def est_shots(a, b):
    calls["n"] += 1                                  # one circuit of 1,000 shots, with its own seed
    return kernel_from_counts(sample_counts(exact_probabilities(overlap_circuit(a, b)), SHOTS_K, seed=calls["n"]))


kernels = {}
for name, est in (("exact", est_exact), ("noisy", est_noisy), ("shots", est_shots)):
    kernels[name] = (np.array(training_kernel(X_train, est)), np.array(test_kernel(X_test, X_train, est)))
print("circuits run for the shot-based kernels:", calls["n"], "  (expected 60)")

fig, ax = plt.subplots(1, 3, figsize=(10, 3.2))
for a, name in zip(ax, kernels):
    im = a.imshow(kernels[name][0], vmin=0, vmax=1, cmap="viridis")
    a.set_title(name); a.set_xticks([]); a.set_yticks([])
fig.colorbar(im, ax=ax, shrink=0.8); plt.show()
off = np.triu_indices(8, 1)
print("largest |noisy - exact| off the diagonal:", round(float(np.max(np.abs(kernels["noisy"][0] - kernels["exact"][0])[off])), 4))
print("largest |shots - noisy| off the diagonal:", round(float(np.max(np.abs(kernels["shots"][0] - kernels["noisy"][0])[off])), 4))

**What to notice.** The three matrices look almost the same: the same two blocks of similar points (the first four points are one class, the last four the other). The noise shifts entries by up to about 0.07, and the shot noise adds a few hundredths more. Whether that matters depends on the classifier, which is the next step.

## Step 5: train and test

The **support vector classifier** (SVC) of Module 4 accepts a precomputed kernel: `SVC(kernel="precomputed", C=1).fit(K_train, y_train)` and then `.score(K_test, y_test)` gives the fraction of test points classified correctly. For comparison, a classical SVC with the **RBF kernel** works on the raw features. Run the cell.

In [ ]:
print(f"{'kernel':>22s}{'training accuracy':>19s}{'test accuracy (4 points)':>26s}")
models = {}
for name, (Ktr, Kte) in kernels.items():
    models[name] = SVC(kernel="precomputed", C=1).fit(Ktr, y_train)
    print(f"{'quantum, ' + name:>22s}{models[name].score(Ktr, y_train):19.2f}{models[name].score(Kte, y_test):26.2f}")
rbf = SVC(kernel="rbf", C=1).fit(X_train, y_train)
print(f"{'classical RBF':>22s}{rbf.score(X_train, y_train):19.2f}{rbf.score(X_test, y_test):26.2f}")
print("\nDecision values on the test points (sign = class, size = margin):")
for name, (Ktr, Kte) in kernels.items():
    print(f"   {name:>6s}:", np.round(models[name].decision_function(Kte), 3), "  true classes", y_test.tolist())

**What to notice.** All three quantum kernels classify the 4 test points correctly, and the decision values are close to each other: the noise and the shots barely move them. The classical RBF kernel gets only 0.25. This data set was **made** from the ZZ feature map, so its classes are simple in the quantum feature space and twisted in the raw features: it is a test of the method, not evidence that quantum kernels beat classical ones on real data.

## Step 6: why the noise does so little, and what 4 points can show

**The noise acts almost like a straight line.** Plot each noisy entry against the exact one: they lie close to a line, K_noisy ≈ a · K_exact + b. Depolarizing noise mixes the state with the completely mixed one, whose overlap is 1/4, so it shrinks every overlap toward a constant. An SVC is not affected by such a change:

- adding a constant b to every entry changes nothing, because the SVC's weights satisfy Σ αᵢ yᵢ = 0, so the constant cancels;
- multiplying by a > 0 is the same as changing C to C / a: the decision values are identical.

The one part that is not a straight line is the **diagonal**, set to 1 instead of the noisy ≈ 0.897. That adds a small constant to the diagonal only, which acts like a little extra regularization.

**Four test points.** A test accuracy from 4 points is a fraction of 4, with σ = √(p (1 − p) / 4): for a true accuracy of 0.9 that is **0.15**. And a classifier that guesses at random gets all 4 right with probability (1/2)⁴ = **0.0625**. So 4 out of 4 is encouraging but weak evidence. Do not put the observed 4 of 4 into the formula: p = 1 gives σ = 0, a certainty that 4 points cannot give. Report "4 of 4" and say how few points that is. If you need an interval, use one made for small counts, such as the **Wilson interval**: for 4 of 4 its 95% range is 0.51 to 1.00. The usual interval p ± 1.96 √(p (1 − p) / N), the normal approximation, behaves poorly for so few points. Run the cell: it fits the line, shows that the SVC on a K + b with C / a gives the same decision values, and then tests the classifiers on all 20 test points of the data set (exact kernels, no shots), which a real run would pay for with 160 more circuits.

In [ ]:
Ke, Kn = kernels["exact"][0], kernels["noisy"][0]
a, b = np.polyfit(Ke[off], Kn[off], 1)
print(f"noisy = {a:.3f} x exact + {b:.3f};  spread around the line {np.std(Kn[off] - (a * Ke[off] + b)):.4f}")
plt.figure(figsize=(4.2, 3.6))
plt.plot(Ke[off], Kn[off], "o", ms=4, label="training entries")
plt.plot([0, 1], [b, a + b], "k--", lw=1, label=f"{a:.3f} K + {b:.3f}")
plt.xlabel("exact K"); plt.ylabel("noisy K"); plt.legend(); plt.tight_layout(); plt.show()

m1 = SVC(kernel="precomputed", C=1).fit(Ke, y_train)
m2 = SVC(kernel="precomputed", C=1 / a).fit(a * Ke + b, y_train)
print("decision values, exact kernel:       ", np.round(m1.decision_function(kernels["exact"][1]), 4))
print("decision values, a K + b with C / a: ", np.round(m2.decision_function(a * kernels["exact"][1] + b), 4))

print("\nWith 4 test points, sigma of an accuracy of 0.9:", round(fraction_sigma(0.9, 4), 3),
      "  chance of 4/4 by guessing:", 0.5 ** 4)


def wilson(k, n, z=1.96):
    """95% Wilson interval for k successes in n trials: better than p +- 1.96 sigma when n is small."""
    p = k / n
    d = 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / d, z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return round(float(max(0.0, c - h)), 2), round(float(min(1.0, c + h)), 2)


print("95% Wilson interval for 4 of 4:", wilson(4, 4))
X_all, y_all = np.array(DATA["X_test"]), np.array(DATA["y_test"])
for name, est in (("exact", est_exact), ("noisy", est_noisy)):
    acc = SVC(kernel="precomputed", C=1).fit(kernels[name][0], y_train).score(test_kernel(X_all, X_train, est), y_all)
    k = int(round(acc * len(y_all)))
    print(f"quantum, {name}: {k} of {len(y_all)} test points right, 95% Wilson interval {wilson(k, len(y_all))}")
k = int(round(rbf.score(X_all, y_all) * len(y_all)))
print(f"classical RBF:  {k} of {len(y_all)} test points right, 95% Wilson interval {wilson(k, len(y_all))}")
pred_exact = SVC(kernel="precomputed", C=1).fit(kernels["exact"][0], y_train).predict(test_kernel(X_all, X_train, est_exact))
pred_q = SVC(kernel="precomputed", C=1).fit(kernels["noisy"][0], y_train).predict(test_kernel(X_all, X_train, est_noisy))
print(f"noisy and exact kernels give the same {len(y_all)} predictions: {bool(np.all(pred_q == pred_exact))}")
pred_r = rbf.predict(X_all)
only_q = int(np.sum((pred_q == y_all) & (pred_r != y_all)))
only_r = int(np.sum((pred_q != y_all) & (pred_r == y_all)))
print(f"paired, same {len(y_all)} points: noisy quantum right and RBF wrong on {only_q}, RBF right and noisy quantum wrong on {only_r}")

**What to notice.** The noisy entries follow K_noisy ≈ 0.852 K_exact + 0.046 closely, and the SVC trained on a K + b with C / a gives exactly the same decision values as the exact kernel: the noise that looks like a straight line is invisible to the classifier. On all 20 test points the quantum kernels reach 0.90 (exact) and 0.90 (noisy), against 0.60 for the RBF kernel: the 4-point result of 1.00 was optimistic. These accuracies describe this test set. Their uncertainty comes from which 20 points happen to be in it, not from the shots, which here are not used at all: 18 of 20 has a 95% Wilson interval of 0.70 to 0.97.

**Comparing the two classifiers.** Both are tested on the **same** 20 points, so their results are paired, not independent, and combined_sigma does not apply to the difference of their accuracies. Only the points where they disagree carry information. The cell compares the RBF kernel with the quantum kernel built from the **noisy** entries, the classifier a run would use (here without shot noise), and first checks that its 20 predictions are identical to those of the exact kernel, so the result holds for both. The noisy quantum kernel is right on 6 points where the RBF kernel is wrong, and the RBF kernel on 0 where the quantum kernel is wrong. If the two were equally good, a split of 6 disagreements as uneven as this would happen by chance with probability about 0.03 (a two-sided sign test). That is suggestive, but weaker than the course's 3σ standard (a chance of about 0.003), and it is one data set built from the same feature map (see Limits). A claim about one particular run with 1,000-shot entries would repeat this comparison with that run's kernel; Step 5 showed its decision values move only a little. Comparing classifiers needs many more test points than a hardware run of this size allows.

## Step 7: your personal check

Open your capstone quiz in Canvas. Question 1 shows your own numbers: PAIR (1 to 28), P2 and READOUT (0.005 to 0.030). Type them below and run the next two cells. The check cell tests the three statistics functions of Step 1 and every function you wrote for this capstone. Only if every test passes does it print your **verification value**: 10,000 times the exact noisy kernel entry of one training pair with your noise model, rounded to a whole number. PAIR numbers the 28 pairs i < j in order: 1 is (0, 1), 2 is (0, 2), ..., 7 is (0, 7), 8 is (1, 2), ..., 28 is (6, 7). For example, PAIR, P2, READOUT = 1, 0.01, 0.02 gives 5583.

In [ ]:
PAIR = 0    # your number from Canvas
P2 = 0    # your number from Canvas
READOUT = 0    # your number from Canvas

In [ ]:
# The course's check code is in l3_m6_common.py and l3_m6_kernel_check.py, next to this notebook.
# It is kept out of the notebook so that the course's reference solutions are not on screen.
# You may open the file to see how your functions are tested, but write your own functions first.
from l3_m6_kernel_check import check_l3_m6_kernel

passed, messages, value = check_l3_m6_kernel(fraction_sigma, expectation_sigma, combined_sigma, overlap_circuit, kernel_from_counts,
                                             training_kernel, test_kernel, PAIR, P2, READOUT)
for m in messages:
    print(m)
if passed:
    print("\nAll tests passed. Your verification value is", value)
else:
    print("\nNot yet: fix the item above and run this cell again.")

## Limits of this experiment

- **Very few points.** 8 training points and 4 test points are what a short run on a real computer allows. They show that the method works, not how well it works: the 20-point test is already more informative, and real data would need far more points and circuits (the training kernel grows as the square of the number of points).
- **A data set made for the method.** The labels come from the same feature map. On real data, a quantum kernel has no known advantage, and a classical kernel is the baseline to beat.
- **The noise model is simple.** Its depolarizing noise acts almost like a straight line on the kernel, which the SVC ignores. Coherent errors and drifting calibrations can distort the kernel in ways that are not a straight line, and the diagonal set to 1 hides how noisy the device is.
- **Two qubits.** With more features, the circuits get deeper and the kernel values of different points get very small (they **concentrate** near 0), so far more shots are needed to tell them apart.
- **No C search.** C = 1 was fixed. With so few points, cross-validation (Module 4) would be unreliable, which is itself a limit of small experiments.

These limits are not failures of your experiment: every experiment has them. Saying what a result does and does not show is part of reporting it.

## Optional: run it on a real IBM quantum computer

If you have an IBM Quantum account with an Open Plan instance (see **Set Up Your Tools** in Start Here), the [Module 6 hardware notebook in Colab](https://colab.research.google.com/github/CSU-Physics/quantum-computing-notebooks/blob/main/colab/level3/QC-L3-M6-hardware-colab.ipynb) runs this capstone's reference circuits on a real computer and compares them with the ideal values and with this noise model. Your API key stays in Colab's **Secrets**; never type it into a cell. The Open Plan gives up to 10 minutes of quantum computer time every 28 days; the hardware notebook uses well under a minute of it, but the queue can take from seconds to hours. This step is not graded, and the badge does not depend on it.

## Your capstone summary (required, not graded)

Write five short answers here (double-click this cell to edit it). They are how a result is reported in a lab notebook or a paper, and they are good preparation for the capstone quiz. Then paste them into **Your Capstone Summary** in the Course Completion module in Canvas: it is not graded, but it is required for the badge.

1. **Question.** What did you want to find out?
2. **Method.** Which circuits, which noise model (P2, READOUT), how many shots, which mitigation or decoding?
3. **Result.** Your main numbers: kernel entries with their uncertainty from the shots (for example 0.558 ± 0.016), and each accuracy as a count of test points (for example 4 of 4, or 18 of 20), with a small-sample interval such as Wilson's if you give one.
4. **Interpretation.** What do the numbers say about the question? For the classifiers, what does the point-by-point comparison on the same test points show? Which kernel entries differ by more than 3σ?
5. **Limits.** What does this experiment not show?

**Next in Canvas:** the **Capstone Quiz — Kernel Classifier**, then the Module 6 time log.